<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/wip-cartpole-dqn-lightning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- TODO: compare dueling dqn vs dqn
- TODO: Replace the two nn.Linear layers in each stream with NoisyLinear; drop ϵ-greedy once training kicks off.
- TODO: Store n-length trajectories in the replay (Tianshou has NStepCollector & NStepPrioritizedReplayBuffer).
- TODO: Swap the single-value head for a categorical head (51 atoms is the canonical choice).
- TODO: make sure to record if training is stopped early (eg: target reached)
- TODO: Log more relevant metrics...
- TODO: Checkpoint the best model via ModelCheckpoint(monitor="reward_mean", save_top_k=1).
- TODO: Add gradient_clip_val=10.0 and precision="16-mixed" to pl.Trainer.

# Solve Gymnasium with Rainbow

This notebook trains a **Deep Q-Network (DQN)** agent on the classic environment using **PyTorch Lightning**.

Install required libraries for Gymnasium, PyTorch Lightning, Tianshou, WandB, and notebook utilities.

In [3]:
%pip install gymnasium[classic-control] pytorch-lightning tianshou wandb[media]>=0.20 tsilva_notebook_utils==0.0.99 > /dev/null

Note: you may need to restart the kernel to use updated packages.


Load API keys and authentication tokens from Colab secrets for secure access.

🔑 Loading API keys and authentication tokens from Colab secrets:

In [4]:
from tsilva_notebook_utils.colab import load_secrets_into_env

_ = load_secrets_into_env([
    'WANDB_API_KEY',
    'NOTIFICATION_URL',
    'NOTIFICATION_AUTH_TOKEN'
])

Define the configuration for the environment and agent hyperparameters.

In [5]:
import torch.nn as nn
from typing import Dict

def setup_config(env_id: str = "CartPole-v1") -> Dict[str, object]:
    # ------------------------------------------------------------------ #
    # Defaults that are reasonable for *most* small, discrete-state tasks
    # ------------------------------------------------------------------ #
    common: Dict[str, object] = dict(
        env_id=env_id,
        seed=42,
        max_epochs=-1,   
        max_steps=50_000,
        log_every_n_steps=10,                   # log every 10 env-steps
        normalize_obs=False,
        buffer_size=10_000,
        per_alpha=0.0,              # 0 → vanilla replay buffer
        per_beta_start=0.4,
        per_beta_end=1.0,
        eval_every_n_episodes=10,
        running_reward_window=100,             # running-mean window
        target_update_interval=200,        # hard-update interval
        target_soft_tau=0.005,                     # soft-update coefficient
        gradient_clip_val=None,        # e.g. 0.5 to enable clipping
        checkpoint_every_n_steps=500,
        train_freq=1,
        gradient_steps=1                     # train every step
    )

    # ------------------------------------------------------------------ #
    # Environment-specific overrides
    # ------------------------------------------------------------------ #
    env_specific: Dict[str, Dict[str, object]] = {
        "CartPole-v1": dict(
            max_steps=50_000,
            hidden_dims=(256, 256),
            discount_factor=0.99,
            batch_size=64,
            buffer_size=100_000,
            learning_starts=1000,
            learning_rate=2.3e-3,
            epsilon_start=1.0,
            epsilon_end=0.04,
            epsilon_decay_steps=25_000,
            exploration_fraction=0.16,
            target_update_interval=10,
            target_soft_tau=0.0,
            train_freq=256,
            gradient_steps=128,
            reward_threshold=475.0
        ),
        "MountainCar-v0": dict(
            hidden_dims=(256, 256),
            discount_factor=0.99,
            batch_size=64,
            learning_starts=5_000,
            buffer_size=100_000,
            learning_rate=2.5e-4,
            epsilon_start=1.0,
            epsilon_end=0.01,
            epsilon_decay_steps=50_000,
            reward_threshold=-110.0
        ),
    }

    if env_id not in env_specific:
        raise ValueError(f"Unsupported env_id: {env_id}")

    # Merge: environment block overrides any duplicate keys in *common*
    return {**common, **env_specific[env_id]}

CONFIG = setup_config()

In [6]:
# --- Runtime metadata --------------------------------------------------------
import subprocess, torch, platform, os

def _get_git_commit() -> str:
    """Return the short SHA if this is a Git repo, else 'unknown'."""
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:          # not a Git checkout or Git not installed
        return "unknown"

def runtime_metadata():
    return {
        "git_commit": _get_git_commit(),
        "torch_version": torch.__version__,
        "torch_cuda": torch.version.cuda or "cpu",
        "cuda_device": (
            torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
        ),
        "python_version": platform.python_version(),
        "run_host": os.uname().nodename,
    }

# Merge into the notebook-wide CONFIG dict
CONFIG.update(runtime_metadata())

In [7]:
# HACK: CartPole-v1 this config achieves 100 reward steadily but slowly (>50k steps)
CONFIG.update(
    learning_rate          = 1e-3,   # or 5e-4
    train_freq             = 1,      # learn every env-step
    gradient_steps         = 1,      # one batch per update
    target_update_interval = 1000,   # hard-update every 1 k env-steps
    gradient_clip_val      = 10.0,   # Lightning's built-in clipping
    buffer_size            = 50_000, # plenty for CartPole
    batch_size             = 64,     # classic
    per_alpha              = 0.0,    # turn PER off until it works
    target_soft_tau=0,
    
    epsilon_start = 1.0,
    epsilon_end   = 0.05,
    epsilon_decay_steps = 25_000

)



In [8]:
CONFIG.update(
    gradient_steps = 2,
    batch_size     = 128,
    target_update_interval = 500,
    learning_rate  = 2e-3,
)


In [9]:
CONFIG

{'env_id': 'CartPole-v1',
 'seed': 42,
 'max_epochs': -1,
 'max_steps': 50000,
 'log_every_n_steps': 10,
 'normalize_obs': False,
 'buffer_size': 50000,
 'per_alpha': 0.0,
 'per_beta_start': 0.4,
 'per_beta_end': 1.0,
 'eval_every_n_episodes': 10,
 'running_reward_window': 100,
 'target_update_interval': 500,
 'target_soft_tau': 0,
 'gradient_clip_val': 10.0,
 'checkpoint_every_n_steps': 500,
 'train_freq': 1,
 'gradient_steps': 2,
 'hidden_dims': (256, 256),
 'discount_factor': 0.99,
 'batch_size': 128,
 'learning_starts': 1000,
 'learning_rate': 0.002,
 'epsilon_start': 1.0,
 'epsilon_end': 0.05,
 'epsilon_decay_steps': 25000,
 'exploration_fraction': 0.16,
 'reward_threshold': 475.0,
 'git_commit': '03f53db',
 'torch_version': '2.5.1',
 'torch_cuda': '11.8',
 'cuda_device': 'NVIDIA GeForce RTX 2060',
 'python_version': '3.11.11',
 'run_host': 'BEAST-2'}

Set the random seed for reproducibility.

In [10]:
from tsilva_notebook_utils.lightning import seed_everything
seed_everything(CONFIG['seed'])

Seed set to 42


42

Login to Weights & Biases (wandb) for experiment tracking.

In [11]:
from wandb import login
login()

wandb: Currently logged in as: tsilva to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Build and test the Gymnasium environment using the provided configuration.

In [12]:
from tsilva_notebook_utils.misc import filter_kwargs
from tsilva_notebook_utils.gymnasium import build_env

env, _, _ = build_env(**filter_kwargs(build_env, CONFIG))
env

<TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v1>>>>>

Initialize the environment and determine the input and output dimensions for the model.

In [13]:
env, _, _ = build_env(**filter_kwargs(build_env, CONFIG))
N_INPUTS = env.observation_space.shape[0]
N_OUTPUTS = int(env.action_space.n)
N_INPUTS, N_OUTPUTS

(4, 2)

Create DQN model:

In [14]:
class DQNModel(nn.Module):
    def __init__(self, n_inputs: int, hidden_dims: tuple[int, ...], n_outputs: int):
        super().__init__()
        layers: list[nn.Module] = []
        last = n_inputs
        for h in hidden_dims:
            layers += [nn.Linear(last, h), nn.ReLU()]
            last = h
        layers.append(nn.Linear(last, n_outputs))
        self.backbone = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)
    
model = DQNModel(N_INPUTS, CONFIG['hidden_dims'], N_OUTPUTS)
model

DQNModel(
  (backbone): Sequential(
    (0): Linear(in_features=4, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=2, bias=True)
  )
)

Create Dueling DQN model:

In [15]:
import torch.nn as nn
import torch

class DuelingDQNModel(nn.Module):
    """
    A feed-forward dueling network:
        • shared feature extractor
        • separate value (V) and advantage (A) streams
        • Q(s,a) = V(s) + A(s,a) − mean_a A(s,a)
    """
    def __init__(self, n_inputs, hidden_dims, n_outputs):
        super().__init__()

        # --- shared feature layers ----------------------------------------
        layers = []
        last = n_inputs
        for h in hidden_dims:
            layers += [nn.Linear(last, h), nn.ReLU()]
            last = h
        self.feature = nn.Sequential(*layers)

        # --- value stream --------------------------------------------------
        self.value = nn.Sequential(
            nn.Linear(last, last),
            nn.ReLU(),
            nn.Linear(last, 1),
        )

        # --- advantage stream ---------------------------------------------
        self.advantage = nn.Sequential(
            nn.Linear(last, last),
            nn.ReLU(),
            nn.Linear(last, n_outputs),
        )

    # ---------------------------------------------------------------------
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not torch.is_tensor(x):                       # allow NumPy inputs
            x = torch.as_tensor(x, dtype=torch.float32)
        f = self.feature(x)
        v = self.value(f)                      # shape: (B, 1)
        a = self.advantage(f)                  # shape: (B, A)
        q = v + a - a.mean(dim=1, keepdim=True)
        return q

model = DuelingDQNModel(N_INPUTS, CONFIG['hidden_dims'], N_OUTPUTS)
model

DuelingDQNModel(
  (feature): Sequential(
    (0): Linear(in_features=4, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
  )
  (value): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=1, bias=True)
  )
  (advantage): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=2, bias=True)
  )
)

Implement the PyTorch Lightning module for DQN training, including replay buffer and training logic.

In [16]:
import random
import numpy as np
import pytorch_lightning as pl
from tianshou.data import Batch, ReplayBuffer, PrioritizedReplayBuffer
from tsilva_notebook_utils.torch import create_infinite_data_loader

infinite_data_loader = create_infinite_data_loader()

class DQNModule(pl.LightningModule):
    def __init__(self, cfg: Dict[str, object], log_basic_every_n: int = 1, log_buffer_every_n: int = 100):
        super().__init__()
        self.cfg = cfg
        self.save_hyperparameters(ignore=["cfg"])  # dump cfg separately
        self.log_basic_every_n = log_basic_every_n
        self.log_buffer_every_n = log_buffer_every_n

        # --- env & nets --------------------------------------------------
        self.build_env_fn = lambda **kwargs: build_env(**filter_kwargs(build_env, {**CONFIG, **kwargs}))
        self.env, self.state, _ = self.build_env_fn()
        n_inputs = self.env.observation_space.shape[0]
        n_outputs = int(self.env.action_space.n)

        self.q_model = DQNModel(n_inputs, cfg["hidden_dims"], n_outputs)
        self.target_q_model = DQNModel(n_inputs, cfg["hidden_dims"], n_outputs)
        self.target_q_model.load_state_dict(self.q_model.state_dict())

        # --- replay buffer ----------------------------------------------
        if cfg["per_alpha"] > 0:
            self.buffer = PrioritizedReplayBuffer(size=cfg["buffer_size"], alpha=cfg["per_alpha"], beta=cfg["per_beta_start"])
        else:
            self.buffer = ReplayBuffer(size=cfg["buffer_size"])

        # --- bookkeeping -------------------------------------------------
        self.total_steps = 0
        self.episode = 0
        self.episode_steps = 0
        self.episode_reward = 0.0
        self.episode_rewards: list[float] = []
        self.random_actions = 0
        self.greedy_actions = 0

    # ------------------------------------------------------------------
    # Convenience properties
    # ------------------------------------------------------------------
    @property
    def eps(self):
        prog = min(1.0, self.total_steps / self.cfg['epsilon_decay_steps'])
        return self.cfg['epsilon_end'] + (self.cfg['epsilon_start'] - self.cfg['epsilon_end']) * (1 - prog)

    @property
    def beta(self) -> float:
        assert self.trainer.max_steps > 0, "Trainer must have max_steps set"
        prog = self.total_steps / (self.cfg['exploration_fraction'] * self.trainer.max_steps)
        prog = min(1.0, prog)
        return self.cfg['per_beta_start'] + prog * (self.cfg['per_beta_end'] - self.cfg['per_beta_start'])

    # ------------------------------------------------------------------
    # Lightning hooks
    # ------------------------------------------------------------------
    def forward(self, x: torch.Tensor):
        return self.q_model(x)

    def configure_optimizers(self):
        return torch.optim.Adam(self.q_model.parameters(), lr=self.cfg["learning_rate"])

    def train_dataloader(self):
        return infinite_data_loader

    # ------------------------------------------------------------------
    # Core RL loop lives in training_step
    # ------------------------------------------------------------------
    def training_step(self, _batch, _batch_idx):  # noqa: N802  (Lightning API)
        # --- choose & execute an action ---------------------------------
        if random.random() < self.eps:
            action, self.random_actions = self.env.action_space.sample(), self.random_actions + 1
        else:
            with torch.no_grad():
                q = self.q_model(torch.as_tensor(self.state, dtype=torch.float32, device=self.device).unsqueeze(0))
            action, self.greedy_actions = int(q.argmax(dim=1).item()), self.greedy_actions + 1

        next_state, reward, term, trunc, info = self.env.step(action)
        done = term or trunc
        self.buffer.add(Batch(
            obs=self.state,
            act=action,
            rew=reward,
            terminated=term,
            truncated=trunc,
            obs_next=next_state,
            info=info,
        ))

        # --- book‑keeping ----------------------------------------------
        self.state = next_state
        self.total_steps += 1
        self.episode_steps += 1
        self.episode_reward += reward

        # --- learn only after warm‑up -----------------------------------
        loss = None
        if len(self.buffer) >= self.cfg["learning_starts"] and self.total_steps % self.cfg["train_freq"] == 0:
            losses = []
            for _ in range(self.cfg["gradient_steps"]):
                l = self._learn_from_buffer()       # one SGD pass
                if l is not None: losses.append(l)
            # Average the losses so Lightning’s single backward() call
            # has a sensible magnitude.
            loss = torch.stack(losses).mean() if losses else None

        # --- per‑step *cheap* metrics -----------------------------------
        if self.total_steps % self.log_basic_every_n == 0:
            self.log_dict({
                "train/eps": self.eps,
                "train/buffer_fill": len(self.buffer) / self.buffer.maxsize,
                "train/reward_step": reward,
            }, on_step=True, prog_bar=False, logger=True)

        # --- end of episode --------------------------------------------
        if done:
            self._on_episode_end()

        return loss  # Lightning will handle backward/optim step

    # ------------------------------------------------------------------
    # Helper: one gradient‑descent update
    # ------------------------------------------------------------------
    def _learn_from_buffer(self):
        # ------- sample --------------------------------------------------
        if self.cfg["per_alpha"] > 0:
            self.buffer.set_beta(self.beta)
            batch, indices = self.buffer.sample(self.cfg["batch_size"])
            weights = torch.as_tensor(batch.weight, dtype=torch.float32, device=self.device)
        else:
            batch, _ = self.buffer.sample(self.cfg["batch_size"])
            weights = torch.ones(self.cfg["batch_size"], dtype=torch.float32, device=self.device)
            indices = None  # type: ignore

        s = torch.as_tensor(batch.obs, dtype=torch.float32, device=self.device)
        a = torch.as_tensor(batch.act, dtype=torch.long, device=self.device).unsqueeze(-1)
        r = torch.as_tensor(batch.rew, dtype=torch.float32, device=self.device)
        s_next = torch.as_tensor(batch.obs_next, dtype=torch.float32, device=self.device)
        d = torch.as_tensor(batch.terminated | batch.truncated, dtype=torch.float32, device=self.device)

        # ------- estimate targets ---------------------------------------
        q_pred = self.q_model(s).gather(1, a).squeeze(1)
        with torch.no_grad():
            next_actions = self.q_model(s_next).argmax(1, keepdim=True)
            q_next = self.target_q_model(s_next).gather(1, next_actions).squeeze(1)
        q_target = r + self.cfg["discount_factor"] * q_next * (1 - d)

        # ------- TD‑error & loss ----------------------------------------
        td_error = q_pred - q_target
        huber = nn.functional.smooth_l1_loss(q_pred, q_target, reduction="none")
        loss = (weights * huber).mean()
        self.log("loss", loss, on_step=True, prog_bar=True)

        # ------- PER priority update ------------------------------------
        if indices is not None:
            priorities = (td_error.detach().abs() + 1e-6) ** self.cfg["per_alpha"]
            self.buffer.update_weight(indices, priorities.cpu().numpy())

        # ------- soft / hard target update ------------------------------
        if self.cfg["target_update_interval"] > 0 and self.total_steps % self.cfg["target_update_interval"] == 0:
            self.target_q_model.load_state_dict(self.q_model.state_dict())
        elif self.cfg["target_soft_tau"] > 0:
            tau = self.cfg["target_soft_tau"]
            with torch.no_grad():
                for t, s in zip(self.target_q_model.parameters(), self.q_model.parameters()):
                    t.copy_(t * (1 - tau) + s * tau)

        # ------- expensive diagnostics (optional) -----------------------
        if self.total_steps % self.log_buffer_every_n == 0:
            self.log_dict({
                "buffer/td_error_mean": td_error.abs().mean(),
                "buffer/q_pred_mean": q_pred.mean(),
                #"buffer/q_pred_max_mean" : q_pred.max(dim=1).values.mean(),
                "buffer/q_target_mean": q_target.mean(),
            }, on_step=True, logger=True)

        return loss

    # ------------------------------------------------------------------
    # Helper: episode wrap‑up
    # ------------------------------------------------------------------
    def _on_episode_end(self):
        self.episode_rewards.append(self.episode_reward)
        last_window = self.episode_rewards[-self.cfg["running_reward_window"] :]
        mean_r = float(np.mean(last_window))

        # episode‑level metrics
        pct_random = self.random_actions / max(1, self.random_actions + self.greedy_actions)
        self.log_dict({
            "episode/episode": self.episode,
            "episode/steps": self.episode_steps,
            "episode/reward": self.episode_reward,
            "episode/reward_mean": mean_r,
            "episode/pct_random": pct_random,
        }, on_step=True, logger=True, prog_bar=True)

        # reset counters
        self.state = self.env.reset()[0]
        self.episode_reward = 0.0
        self.episode_steps = 0
        self.episode += 1
        self.random_actions = 0
        self.greedy_actions = 0

    # ------------------------------------------------------------------
    # Gradient diagnostics – after backward so grads exist
    # ------------------------------------------------------------------
    def on_after_backward(self):  # noqa: D401  (Lightning API)
        grad_norm = torch.nn.utils.clip_grad_norm_(self.q_model.parameters(), max_norm=float("inf"))
        with torch.no_grad():
            param_l2 = sum(p.norm(2) for p in self.q_model.parameters())
        self.log_dict({
            "grad/grad_l2": grad_norm, 
            "grad/param_l2": param_l2
        }, on_step=True, logger=True)

module = DQNModule(CONFIG)
module

DQNModule(
  (q_model): DQNModel(
    (backbone): Sequential(
      (0): Linear(in_features=4, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): ReLU()
      (4): Linear(in_features=256, out_features=2, bias=True)
    )
  )
  (target_q_model): DQNModel(
    (backbone): Sequential(
      (0): Linear(in_features=4, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): ReLU()
      (4): Linear(in_features=256, out_features=2, bias=True)
    )
  )
)

Set up the PyTorch Lightning trainer and start training the DQN agent.

In [17]:
import os
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint

from tsilva_notebook_utils.lightning import StopOnLambda
from tsilva_notebook_utils.gymnasium import build_pl_callback

checkpoint_cb = ModelCheckpoint(
    monitor="episode/reward_mean",
    mode="max",
    save_top_k=1,                    # keep only the best
    every_n_train_steps=CONFIG['checkpoint_every_n_steps'],           # evaluate after *every* step
    save_on_train_epoch_end=False    # don’t wait for epoch end (there is none)
    # optional: dirpath="checkpoints/", filename="dqn-{reward_mean:.2f}-{step}"
)

trainer = pl.Trainer(
    max_epochs=CONFIG['max_epochs'],
    max_steps=CONFIG['max_steps'],
    log_every_n_steps=CONFIG['log_every_n_steps'],
    logger=WandbLogger(project=os.getenv('NOTEBOOK_ID'), config=CONFIG),
    enable_model_summary=False,
    gradient_clip_val=CONFIG['gradient_clip_val'],
    callbacks=[
        checkpoint_cb,
        build_pl_callback("EvalEpisodeAndRecordCallback", every_n_episodes=CONFIG['eval_every_n_episodes']),
        StopOnLambda(
            lambda metrics: metrics.get('reward_mean', -float('inf')) >= CONFIG['reward_threshold'],
            message=f"Stopping: reward_mean >= {CONFIG['reward_threshold']}"
        )
    ]
)
trainer.fit(module)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/tsilva/miniconda3/envs/aiml-notebooks/lib/python3.11/site-packages/wandb/analytics/sentry.py:258: DeprecationWarning: The `Scope.user` setter is deprecated in favor of `Scope.set_user()`.
  self.scope.user = {"email": email}
/home/tsilva/miniconda3/envs/aiml-notebooks/lib/python3.11/site-packages/wandb/analytics/sentry.py:258: DeprecationWarning: The `Scope.user` setter is deprecated in favor of `Scope.set_user()`.
  self.scope.user = {"email": email}


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Training: |          | 0/? [00:00<?, ?it/s]

/home/tsilva/miniconda3/envs/aiml-notebooks/lib/python3.11/site-packages/pytorch_lightning/loops/optimization/automatic.py:134: `training_step` returned `None`. If this was on purpose, ignore this warning...
/home/tsilva/miniconda3/envs/aiml-notebooks/lib/python3.11/site-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
`Trainer.fit` stopped: `max_steps=50000` reached.


Render and visualize a trained episode using the learned Q-network.

In [18]:
from tsilva_notebook_utils.gymnasium import render_episode

render_episode(
    env=module.build_env_fn,
    model=module.q_model
)